# Input + Results + Velocity samenvoegen tot één csv

Combineert `Input_redo.xlsx`, `Results_redo.xlsx` en `Results_velocity_redo.xlsx` tot één platte csv waarbij elke case één rij is. De volgorde van de cases is in alle drie de bestanden identiek aan de input.

In [1]:
import pandas as pd

inp = pd.read_excel('Input_redo.xlsx')
res = pd.read_excel('Results_redo.xlsx')
vel = pd.read_excel('Results_velocity_redo.xlsx')

print('Input shape:', inp.shape)
print('Results shape:', res.shape)
print('Velocity shape:', vel.shape)

Input shape: (880, 14)
Results shape: (2, 7920)
Velocity shape: (1, 5280)


## Results plat slaan
Per case heeft Results 9 kolommen × 2 rijen. Std en Mean staan op rij 0, MPM op rij 1 (in de laatste 3 kolommen). Hier combineer ik dat tot één rij per case.

In [2]:
n_cases = len(inp)
stats = ['Std Z', 'Std pitch', 'Std roll',
         'Mean Z', 'Mean pitch', 'Mean roll',
         'MPM Z', 'MPM pitch', 'MPM roll']

rows = []
for i in range(n_cases):
    block = res.iloc[:, i*9:(i+1)*9]
    row = {}
    for j, stat in enumerate(stats):
        if stat.startswith('MPM'):
            row[stat] = block.iloc[1, j]   # MPM staat in rij 1
        else:
            row[stat] = block.iloc[0, j]   # Std/Mean in rij 0
    rows.append(row)

res_clean = pd.DataFrame(rows)
res_clean.head()

,Std Z,Std pitch,Std roll,Mean Z,Mean pitch,Mean roll,MPM Z,MPM pitch,MPM roll
0,0.014982,0.675174,0.000173,-0.155893,0.684405,-0.002331,-0.095942,3.390856,-0.001673
1,0.014647,0.672426,0.000158,-0.155882,0.684157,-0.002331,-0.097238,3.379648,-0.001730
2,0.014984,0.674639,0.000173,-0.155892,0.684348,-0.002331,-0.095931,3.388490,-0.001674
3,0.014847,0.673434,0.000162,-0.155889,0.684243,-0.002331,-0.096463,3.383665,-0.001712
4,0.011490,0.310805,0.351683,-0.155533,0.675667,0.004330,-0.109827,1.920572,1.411232


## Velocity plat slaan
Per case heeft Velocity 6 kolommen × 1 rij: `Mean velocity Z/pitch/roll` en `std velocity Z/pitch/roll`. Zelfde aanpak: blok per case uitsnijden en tot één rij maken. De cases staan in dezelfde volgorde als de input.

In [3]:
vel_stats = ['Mean velocity Z', 'Mean velocity pitch', 'Mean velocity roll',
             'std velocity Z', 'std velocity pitch', 'std velocity roll']

vrows = []
for i in range(n_cases):
    block = vel.iloc[:, i*6:(i+1)*6]
    vrows.append({stat: block.iloc[0, j] for j, stat in enumerate(vel_stats)})

vel_clean = pd.DataFrame(vrows)
vel_clean.head()

,Mean velocity Z,Mean velocity pitch,Mean velocity roll,std velocity Z,std velocity pitch,std velocity roll
0,0.000204,0.000001,1.022967e-09,0.026532,0.021246,0.000002
1,0.000183,0.000001,9.104538e-10,0.026206,0.021172,0.000002
2,0.000200,0.000001,1.090954e-09,0.026533,0.021231,0.000002
3,0.000191,0.000001,9.518392e-10,0.026401,0.021199,0.000002
4,0.000165,0.000049,4.245085e-05,0.018765,0.009632,0.010678


## Samenvoegen en wegschrijven

In [4]:
# Strip de %-prefix uit de inputkolomnamen
inp = inp.rename(columns=lambda c: c.lstrip('%'))

combined = pd.concat([inp.reset_index(drop=True), res_clean, vel_clean], axis=1)
combined.to_csv('Simulation_output_redo.csv', index=False)

print('Shape:', combined.shape)
combined.head()

Shape: (880, 29)


,filename,Hs,Tp,Tz,Direction,Damp_Heave,Damp_Roll,Damp_Pitch,Lin_Heave,Quad_Heave,...,Mean roll,MPM Z,MPM pitch,MPM roll,Mean velocity Z,Mean velocity pitch,Mean velocity roll,std velocity Z,std velocity pitch,std velocity roll
0,ymlfiles\Case_001_H1_T3p5_dir0_heavelow_pitchl...,1,3.5,2.7132,0,low,low,low,1.0,1.0,...,-0.002331,-0.095942,3.390856,-0.001673,0.000204,0.000001,1.022967e-09,0.026532,0.021246,0.000002
1,ymlfiles\Case_002_H1_T3p5_dir0_heavelinhigh_pi...,1,3.5,2.7132,0,linhigh,linhigh,linhigh,6.5,1.0,...,-0.002331,-0.097238,3.379648,-0.001730,0.000183,0.000001,9.104538e-10,0.026206,0.021172,0.000002
2,ymlfiles\Case_003_H1_T3p5_dir0_heavequadhigh_p...,1,3.5,2.7132,0,quadhigh,quadhigh,quadhigh,1.0,4.0,...,-0.002331,-0.095931,3.388490,-0.001674,0.000200,0.000001,1.090954e-09,0.026533,0.021231,0.000002
3,ymlfiles\Case_004_H1_T3p5_dir0_heaveDOFbest_pi...,1,3.5,2.7132,0,DOFbest,DOFbest,DOFbest,3.3,2.4,...,-0.002331,-0.096463,3.383665,-0.001712,0.000191,0.000001,9.518392e-10,0.026401,0.021199,0.000002
4,ymlfiles\Case_005_H1_T3p5_dir45_heavelow_pitch...,1,3.5,2.7132,45,low,low,low,1.0,1.0,...,0.004330,-0.109827,1.920572,1.411232,0.000165,0.000049,4.245085e-05,0.018765,0.009632,0.010678
